# AI Inventory Intelligence & Supplier Orchestration 📦🛡️
Run the full Streamlit Dashboard directly from Google Colab!

**Instructions:**
1. Create a free account at [ngrok.com](https://ngrok.com/) to get an Auth Token.
2. Run the first cell to setup the environment.
3. Add your API keys and **Ngrok Token** in the second cell.
4. Run the final cell to launch the reliable tunnel.

In [ ]:
!git clone https://github.com/lmudu2/ai-inventory-intelligence.git
%cd ai-inventory-intelligence
!pip install streamlit pandas plotly langgraph langchain-groq langchain-core groq python-dotenv requests numpy sendgrid pyngrok

In [ ]:
import os
# ⚠️ ENTER YOUR STAGING API KEYS HERE ⚠️
os.environ['GROQ_API_KEY'] = 'gsk_...'
os.environ['SENDGRID_API_KEY'] = 'SG...'
os.environ['SENDGRID_FROM_EMAIL'] = 'your_email@domain.com'
os.environ['SENDGRID_TO_EMAIL'] = 'target_email@domain.com'

# ⚠️ NGROK AUTH TOKEN (Required for stable tunneling) ⚠️
# Get your free token at https://dashboard.ngrok.com/get-started/your-authtoken
os.environ['NGROK_AUTH_TOKEN'] = 'your_authtoken_here'

print("Environment Variables Set!")

In [ ]:
import time, socket
from pyngrok import ngrok
import os

def wait_for_port(port, timeout=40):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=1):
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                return False
            time.sleep(1)

print("🧹 Cleaning up existing processes...")
ngrok.kill() # Terminate any existing ngrok processes
!fuser -k 8501/tcp

print("🚀 Launching AI Supply Chain Dashboard...")
get_ipython().system_raw("streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &")

if wait_for_port(8501):
    print("✅ Dashboard Backend Ready!")
    
    ngrok_token = os.environ.get('NGROK_AUTH_TOKEN')
    if not ngrok_token or ngrok_token == 'your_authtoken_here':
        print("❌ ERROR: You must provide a valid NGROK_AUTH_TOKEN in the environment cell.")
    else:
        try:
            ngrok.set_auth_token(ngrok_token)
            public_url = ngrok.connect(8501)
            print("\n--- 🔗 UNIVERSAL ACCESS BRIDGE (ngrok) ---")
            print(f"Dashboard URL: {public_url.public_url}")
            print("-----------------------------------------\n")
        except Exception as e:
            print(f"❌ Failed to start ngrok tunnel: {e}")
else:
    print("❌ Startup failed. Displaying logs:")
    !cat streamlit.log